# GitHub Repository 기반 Q&A 챗봇


In [19]:
from dotenv import load_dotenv

load_dotenv()

True

LangChain 추적


In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Code Analysis"

### 데이터 준비


GitHub Repository 를 다운로드 받습니다. 이번 예제는 langchain-ai 공식 Repository 로 진행합니다.

- !git clone 을 사용하여 Repository clone

다른 저장소의 파일을 사용하고 싶다면, `root_dir`을 여러분의 저장소의 루트 디렉토리로 변경하세요.


저는 `/Users/teddy/Dev/github/langchain` 위치에 `langchain` reposotory 를 clone 하였습니다. 아래의 경로는 본인의 경로에 맞게 바꾸어 주어야 합니다.


In [2]:
!dir "D:\Programming\temp\langchain\libs"

 D ����̺��� ����: �� ����
 ���� �Ϸ� ��ȣ: 5AC1-5438

 D:\Programming\temp\langchain\libs ���͸�

2024-06-30  ���� 07:32    <DIR>          .
2024-06-30  ���� 07:32    <DIR>          ..
2024-06-30  ���� 07:32    <DIR>          cli
2024-06-30  ���� 07:32    <DIR>          community
2024-06-30  ���� 07:32    <DIR>          core
2024-06-30  ���� 07:32    <DIR>          experimental
2024-06-30  ���� 07:32    <DIR>          langchain
2024-06-30  ���� 07:32    <DIR>          partners
2024-06-30  ���� 07:32    <DIR>          standard-tests
2024-06-30  ���� 07:32    <DIR>          text-splitters
               0�� ����                   0 ����Ʈ
              10�� ���͸�  122,625,892,352 ����Ʈ ����


## 도큐먼트 로더


여기서 모든 패키지의 파일을 불러오지 않습니다. 핵심 기능을 포함하는 특정 폴더의 파일만 불러오도록 아래와 같이 정의해 주었습니다.


In [3]:
import os
# Root 경로
repo_root = "D:\\Programming\\temp\\langchain\\libs"

# 불러오고자 하는 패키지 경로
repo_core = os.path.join(repo_root, "core", "langchain_core")
repo_community = os.path.join(repo_root, "community", "langchain_community")
repo_experimental = os.path.join(repo_root, "experimental", "langchain_experimental")
repo_parters = os.path.join(repo_root, "partners")
repo_text_splitter = os.path.join(repo_root, "text_splitters", "langchain_text_splitters")
repo_cookbook = "D:\\Programming\\temp\\langchain\\cookbook"  # repo_root + "/cookbook"


In [18]:
# langchain의 여러 모듈을 가져옵니다.
from langchain_text_splitters import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser

# 불러온 문서를 저장할 빈 리스트를 생성합니다.
py_documents = []

for path in [repo_core, repo_community, repo_experimental, repo_parters, repo_cookbook]:
    # GenericLoader를 사용하여 파일 시스템에서 문서를 로드합니다.
    loader = GenericLoader.from_filesystem(
        path,  # 문서를 불러올 경로
        glob="**/*.py",  # 모든 하위 폴더와 파일을 대상으로 함
        # suffixes=[".py"],  # .py 확장자를 가진 파일만 대상으로 함
        parser=LanguageParser(
            language=Language.PYTHON, parser_threshold=30
        ),  # 파이썬 언어의 문서를 파싱하기 위한 설정
    )
    # 로더를 통해 불러온 문서들을 documents 리스트에 추가합니다.
    py_documents.extend(loader.load())

print(f".py 파일의 개수: {len(py_documents)}")

.py 파일의 개수: 5831


In [4]:
import os
from glob import glob
from langchain_text_splitters import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.document_loaders import NotebookLoader

# 불러온 문서를 저장할 빈 리스트를 생성합니다.
py_documents = []
ipynb_documents = []

for path in [repo_core, repo_community, repo_experimental, repo_parters, repo_cookbook]:
    # GenericLoader를 사용하여 파일 시스템에서 문서를 로드합니다.
    # .py 파일 처리
    loader = GenericLoader.from_filesystem(
        path,  # 문서를 불러올 경로
        glob="**/*.py",  # 모든 하위 폴더와 파일을 대상으로 함
        parser=LanguageParser(
            language=Language.PYTHON, parser_threshold=30
        ),  # 파이썬 언어의 문서를 파싱하기 위한 설정
    )
    # 로더를 통해 불러온 문서들을 documents 리스트에 추가합니다.
    py_documents.extend(loader.load())

 # .ipynb 파일 처리
    for notebook_path in glob(os.path.join(path, "**/*.ipynb"), recursive=True):
        try:
            notebook_loader = NotebookLoader(notebook_path, include_outputs=False, max_output_length=10)
            ipynb_documents.extend(notebook_loader.load())
        except PermissionError:
            print(f"Permission denied for file: {notebook_path}")
        except Exception as e:
            print(f"Error processing file {notebook_path}: {str(e)}")

print(f".py 파일의 개수: {len(py_documents)}")
print(f".ipynb 파일의 개수: {len(ipynb_documents)}")

.py 파일의 개수: 5831
.ipynb 파일의 개수: 95


다음은 `.mdx` 확장자를 가진 파일을 `TextLoader` 를 사용하여 불러옵니다. `.mdx` 파일은 Jupyter Notebook 파일을 마크다운 형식으로 변환한 파일이며, 유용한 예제를 포함하고 있으므로 이를 DB 에 추가하기 위해 도큐먼트 형식으로 로드압니다.


In [5]:
import os

# TextLoader 모듈을 불러옵니다.
from langchain_community.document_loaders import TextLoader

# 검색할 최상위 디렉토리 경로를 정의합니다.
# root_dir = "/Users/teddy/Dev/github/langchain/"
root_dir = "D:\\Programming\\temp\\langchain\\"

mdx_documents = []
# os.walk를 사용하여 root_dir부터 시작하는 모든 디렉토리를 순회합니다.
for dirpath, dirnames, filenames in os.walk(root_dir):
    # 각 디렉토리에서 파일 목록을 확인합니다.
    for file in filenames:
        # 파일 확장자가 .mdx인지 확인하고, 경로 내 '*venv/' 문자열이 포함되지 않는지도 체크합니다.
        if (file.endswith(".mdx")) and "*venv/" not in dirpath:
            try:
                # TextLoader를 사용하여 파일의 전체 경로를 지정하고 문서를 로드합니다.
                loader = TextLoader(os.path.join(dirpath, file), encoding="utf-8")
                # 로드한 문서를 분할하여 documents 리스트에 추가합니다.
                mdx_documents.extend(loader.load())
            except Exception:
                # 파일 로드 중 오류가 발생하면 이를 무시하고 계속 진행합니다.
                pass

# 최종적으로 불러온 문서의 개수를 출력합니다.
print(f".mdx 파일의 개수: {len(mdx_documents)}")

.mdx 파일의 개수: 277


## Chunk 분할


파일들을 청크로 나누어 봅시다.


In [25]:
# RecursiveCharacterTextSplitter 모듈을 가져옵니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter 객체를 생성합니다. 이 때, 파이썬 코드를 대상으로 하며,
# 청크 크기는 2000, 청크간 겹치는 부분은 200 문자로 설정합니다.
py_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2000, chunk_overlap=200
)

# py_docs 변수에 저장된 문서들을 위에서 설정한 청크 크기와 겹치는 부분을 고려하여 분할합니다.
py_docs = py_splitter.split_documents(py_documents)

# 분할된 텍스트의 개수를 출력합니다.
print(f"분할된 .py 파일의 개수: {len(py_docs)}")

mdx_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200)

# mdx_docs 변수에 저장된 문서들을 위에서 설정한 청크 크기와 겹치는 부분을 고려하여 분할합니다.
mdx_docs = mdx_splitter.split_documents(mdx_documents)

# 분할된 텍스트의 개수를 출력합니다.
print(f"분할된 .mdx 파일의 개수: {len(mdx_docs)}")

분할된 .py 파일의 개수: 10377
분할된 .mdx 파일의 개수: 634


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

# Python 파일을 위한 splitter
py_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=2000, chunk_overlap=200
)

# Python 문서 분할
py_docs = py_splitter.split_documents(py_documents)
print(f"분할된 .py 파일의 개수: {len(py_docs)}")

# MDX 파일을 위한 splitter
mdx_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200
)

# MDX 문서 분할
mdx_docs = mdx_splitter.split_documents(mdx_documents)
print(f"분할된 .mdx 파일의 개수: {len(mdx_docs)}")

# Jupyter Notebook 파일을 위한 splitter
# 마크다운과 코드를 모두 고려해야 하므로, 별도의 설정을 사용합니다.
ipynb_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,  # Python 코드 셀을 위한 설정
    chunk_size=2000,
    chunk_overlap=200
)

# Jupyter Notebook 문서 분할
ipynb_docs = ipynb_splitter.split_documents(ipynb_documents)
print(f"분할된 .ipynb 파일의 개수: {len(ipynb_docs)}")

# 모든 분할된 문서를 하나의 리스트로 결합
all_docs = py_docs + mdx_docs + ipynb_docs
print(f"총 분할된 문서의 개수: {len(all_docs)}")

분할된 .py 파일의 개수: 10377
분할된 .mdx 파일의 개수: 634
분할된 .ipynb 파일의 개수: 549
총 분할된 문서의 개수: 11560


로드한 도큐먼트를 합칩니다.


In [7]:
combined_documents = py_docs + mdx_docs + ipynb_docs
print(f"총 도큐먼트 개수: {len(combined_documents)}")

총 도큐먼트 개수: 11560


## Embedding


In [10]:
# langchain_openai와 langchain의 필요한 모듈들을 가져옵니다.
from langchain_openai import OpenAIEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

# 로컬 파일 저장소를 사용하기 위해 LocalFileStore 인스턴스를 생성합니다.
# './cache/' 디렉토리에 데이터를 저장합니다.
store = LocalFileStore("./cache/")

# OpenAI 임베딩 모델 인스턴스를 생성합니다. 모델명으로 "text-embedding-3-small"을 사용합니다.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", disallowed_special=())

# CacheBackedEmbeddings를 사용하여 임베딩 계산 결과를 캐시합니다.
# 이렇게 하면 임베딩을 여러 번 계산할 필요 없이 한 번 계산된 값을 재사용할 수 있습니다.
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, store, namespace=embeddings.model
)

In [8]:
# 오픈소스 모델 (Sentence Transformers) 사용
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

# 로컬 파일 저장소 설정
store = LocalFileStore("./cache/")

# Sentence Transformers 모델 설정 (예: all-MiniLM-L6-v2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 캐시 백엔드 임베딩 설정
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, store, namespace=embeddings.model_name
)

# 사용 예시
text = "This is a sample text for embedding."
embedded = cached_embeddings.embed_query(text)
print(f"Embedded dimension: {len(embedded)}")

c:\Users\28006030\.conda\envs\langchain\lib\site-packages\langchain_core\_api\deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
c:\Users\28006030\.conda\envs\langchain\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
c:\Users\28006030\.conda\envs\langchain\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you w

Embedded dimension: 384


c:\Users\28006030\.conda\envs\langchain\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [49]:
import os
# Claude.ai 사용 (Anthropic API 키 필요)
import anthropic
# from langchain.embeddings import AnthropicEmbeddings
from langchain.embeddings import AnthropicEmbeddings, CacheBackedEmbeddings # langchain.embeddings에서 AnthropicEmbeddings import
from langchain.storage import LocalFileStore

# 로컬 파일 저장소 설정
store = LocalFileStore("./cache/")

# Claude 임베딩 설정
embeddings = AnthropicEmbeddings(
    model= "claude-3-5-sonnet-20240620",  # 또는 사용 가능한 최신 Claude 모델
    anthropic_api_key= os.environ["CLAUDE_API_KEY"]# "your_anthropic_api_key_here"  # 실제 API 키로 교체 필요
)

# 캐시 백엔드 임베딩 설정
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, store, namespace="claude-embeddings"
)

# 사용 예시
text = "This is a sample text for embedding."
embedded = cached_embeddings.embed_query(text)
print(f"Embedded dimension: {len(embedded)}")


ModuleNotFoundError: No module named 'anthropic'

## Vector DB


- `from_documents`: 이 메서드는 문서 컬렉션과 이에 해당하는 임베딩을 받아 벡터 인덱스를 생성합니다. 여기서는 `combined_documents` 로부터 벡터를 생성하고, 이 벡터들은 `cached_embeddings` 을 통해 임베딩된 데이터를 사용합니다.
- `save_local`: 이 메서드는 생성된 FAISS 인덱스를 지정된 로컬 폴더에 저장합니다. 이 폴더명은 `FAISS_DB_INDEX` 변수에 저장되어 있습니다.


In [10]:
# langchain_community 모듈에서 FAISS 클래스를 가져옵니다.
from langchain_community.vectorstores import FAISS

# 로컬에 저장할 FAISS 인덱스의 폴더 이름을 지정합니다.
FAISS_DB_INDEX = "langchain_faiss"

# combined_documents 문서들과 cached_embeddings 임베딩을 사용하여
# FAISS 데이터베이스 인스턴스를 생성합니다.
db = FAISS.from_documents(combined_documents, cached_embeddings)

# 생성된 데이터베이스 인스턴스를 지정한 폴더에 로컬로 저장합니다.
db.save_local(folder_path=FAISS_DB_INDEX)

저장한 FAISS 데이터베이스를 불러옵니다. 이후 실행시에는 새롭게 DB 에 저장할 필요없이 아래 코드만 실행하면 됩니다.


In [11]:
# langchain_community 모듈에서 FAISS 클래스를 가져옵니다.
from langchain_community.vectorstores import FAISS

# FAISS 클래스의 load_local 메서드를 사용하여 저장된 벡터 인덱스를 로드합니다.
db = FAISS.load_local(
    FAISS_DB_INDEX,  # 로드할 FAISS 인덱스의 디렉토리 이름
    cached_embeddings,  # 임베딩 정보를 제공
    allow_dangerous_deserialization=True,  # 역직렬화를 허용하는 옵션
)

## Retriever


In [12]:
# MMR을 사용하여 검색을 수행하는 retriever를 생성합니다.
faiss_retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 10})

In [14]:
# langchain.retrievers 모듈에서 BM25Retriever 클래스를 가져옵니다.
from langchain.retrievers import BM25Retriever

# 문서 컬렉션을 사용하여 BM25 검색 모델 인스턴스를 생성합니다.
bm25_retriever = BM25Retriever.from_documents(
    combined_documents  # 초기화에 사용할 문서 컬렉션
)

# BM25Retriever 인스턴스의 k 속성을 10으로 설정하여,
# 검색 시 최대 10개의 결과를 반환하도록 합니다.
bm25_retriever.k = 10

In [15]:
# langchain.retrievers 모듈에서 EnsembleRetriever 클래스를 가져옵니다.
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever 인스턴스를 생성합니다.
# 이때, BM25 검색 모델과 FAISS 검색 모델을 결합하여 사용합니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],  # 사용할 검색 모델의 리스트
    weights=[0.6, 0.4],  # 각 검색 모델의 결과에 적용할 가중치
    search_type="mmr",  # 검색 결과의 다양성을 증진시키는 MMR 방식을 사용
)

## 파이프라인 연결

chain 을 구성합니다.


### 프롬프트


In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 20년차 AI 개발자입니다. 당신의 임무는 주어진 질문에 대하여 최대한 문서의 정보를 활용하여 답변하는 것입니다.
문서는 Python 코드에 대한 정보를 담고 있습니다. 따라서, 답변을 작성할 때에는 Python 코드에 대한 상세한 code snippet을 포함하여 작성해주세요.
최대한 자세하게 답변하고, 한글로 답변해 주세요. 주어진 문서에서 답변을 찾을 수 없는 경우, "문서에 답변이 없습니다."라고 답변해 주세요.
답변은 출처(source)를 반드시 표기해 주세요.

#참고문서:
{context}

#질문:
{question}

#답변: 

출처:
- source1
- source2
- ...                             
"""
)

### ChatLMstudio, ChatGemini 직접 구현

In [ ]:
from typing import Any, List, Mapping, Optional
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, ChatMessage, HumanMessage, AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult

# LM Studio 및 Gemini API 연동 로직 (실제 API 키 및 엔드포인트 정보 필요)
class ChatLMstudio(BaseChatModel):
    model: str
    temperature: float = 0
    streaming: bool = False
    callbacks: List[Any] = []

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        try:
            # LM Studio API 호출 (실제 구현 필요)
            response = lm_studio_api_call(self.model, messages, self.temperature, stop)
            return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response))])
        except Exception as e:
            raise Exception(f"LM Studio API call failed: {e}")

    @property
    def _llm_type(self) -> str:
        return "lm_studio"

class ChatGemini(BaseChatModel):
    model: str
    temperature: float = 0
    streaming: bool = False
    callbacks: List[Any] = []

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        try:
            # Gemini API 호출 (실제 구현 필요)
            response = gemini_api_call(self.model, messages, self.temperature, stop)
            return ChatResult(generations=[ChatGeneration(message=AIMessage(content=response))])
        except Exception as e:
            raise Exception(f"Gemini API call failed: {e}")

    @property
    def _llm_type(self) -> str:
        return "gemini"

### LLM 정의


In [21]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.runnables import ConfigurableField
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_openai import ChatOpenAI
from langchain.chat_models import ChatAnthropic
from langchain_community.chat_models import ChatOllama

class StreamCallback(BaseCallbackHandler):
    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)

llm = ChatOpenAI(
    model="gpt-4-turbo-preview",
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()], # StreamCallback 대신 StreamingStdOutCallbackHandler 사용
).configurable_alternatives(
    ConfigurableField(id="llm"),
    default_key="gpt4",
    claude=ChatAnthropic(
        model="claude-3-5-sonnet-20240620", # "claude-3-opus-20240229",
        temperature=0,
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()],
    ),
    gpt3=ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()],
    ),
    ollama=ChatOllama( # ChatOllama에도 streaming 설정 추가
        model="EEVE-Korean-Instruct-10.8B:latest",
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()],
    ),
    lmstudio = ChatOpenAI(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        model="cognitivecomputations/dolphin-2.9-llama3-8b-gguf",
        temperature=0,
        streaming=True,
        callbacks=[StreamingStdOutCallbackHandler()],
    ),
    # gemini=ChatGemini(
    #     model="gemini-pro",
    #     temperature=0,
    #     streaming=True,
    #     callbacks=[StreamingStdOutCallbackHandler()],
    # ),  
)

In [22]:
# 체인을 생성합니다.
rag_chain = (
    {"context": ensemble_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 질의-응답 테스트


In [23]:
answer = rag_chain.with_config(configurable={"llm": "lmstudio"}).invoke(
    "PromptTemplate 사용방법을 알려주세요"
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [21]:
answer = rag_chain.with_config(configurable={"llm": "ollama"}).invoke(
    "PromptTemplate 사용방법을 알려주세요"
)

Python의 Langchain 라이브러리는 자연어 처리(NLP) 작업에 사용할 수 있는 다양한 도구와 유틸리티를 제공합니다. PromptTemplate은 사용자가 모델에게 입력할 메시지를 생성하는 데 도움을 주는 클래스입니다. 이 클래스는 문자열 형식을 사용하여 사용자 정의 가능한 템플릿을 만들 수 있게 해줍니다.

PromptTemplate을 사용하려면 먼저 Langchain 라이브러리를 설치해야 합니다:
```bash
pip install langchain
```
설치 후, 다음 코드를 사용하여 PromptTemplate 인스턴스를 생성할 수 있습니다:
```python
from langchain import PromptTemplate

# 템플릿 정의
template = "What is the capital of {country}?"

# PromptTemplate 인스턴스 생성
prompt_template = PromptTemplate(template=template)

# 변수 추가
input_variables = {"country": "France"}
partial_prompt = prompt_template.partial(**input_variables)

# 완성된 템플릿 출력
print(partial_prompt)
```
이 예제에서 `"What is the capital of {country}?"`라는 문자열 형식을 가진 템플릿을 정의했습니다. 그런 다음 `PromptTemplate` 클래스를 사용하여 이 템플릿으로 인스턴스를 생성하고, `partial()` 메서드를 사용하여 변수(예: "France")를 추가하여 완성된 템플릿을 만들었습니다. 마지막으로 `print()` 함수를 사용하여 출력물을 표시했습니다.

Langchain 라이브러리는 또한 PromptTemplate과 함께 사용할 수 있는 여러 유틸리티와 클래스를 제공합니다. 예를 들어, `BasePromptTemplate` 클래스는 사용자가 모델에게 입력할 메시지를 생성하는 데 도

In [22]:
answer = rag_chain.with_config(configurable={"llm": "claude"}).invoke(
    "PromptTemplate 사용방법을 알려주세요"
)

PromptTemplate을 사용하는 방법은 다음과 같습니다:

1. PromptTemplate 임포트하기
```python
from langchain.prompts import PromptTemplate
```

2. PromptTemplate 인스턴스 생성하기
- template 인자에 프롬프트 템플릿 문자열을 전달합니다. 
- 템플릿에서 변수는 중괄호({})로 표시합니다.
```python
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}?") 
```

3. PromptTemplate 포맷팅하기
- format() 메소드를 호출하여 프롬프트 템플릿의 변수에 값을 채웁니다.
```python
prompt.format(product="colorful socks")
```

4. (선택사항) 프롬프트 템플릿 파셜링하기 
- 프롬프트 템플릿의 일부 변수만 지정하고 싶다면 partial() 메소드를 사용할 수 있습니다.
  
5. (선택사항) 프롬프트 템플릿 합성하기
- 여러 개의 프롬프트 템플릿을 합쳐 하나의 프롬프트를 만들 수 있습니다.

PromptTemplate을 사용하면 사용자 입력값을 모델에 그대로 전달하는 대신, 입력값을 적절한 프롬프트 형식으로 가공할 수 있습니다. 이는 언어 모델을 활용한 애플리케이션 개발 시 유용하게 활용될 수 있습니다.

출처:
- /Users/teddy/Dev/github/langchain/docs/docs/modules/model_io/index.mdx
- /Users/teddy/Dev/github/langchain/libs/core/langchain_core/prompts/string.py

In [23]:
answer = rag_chain.invoke("WebbaseLoader 사용법에 대해 알려주세요")

`WebBaseLoader`는 HTML 페이지를 로드하고 `BeautifulSoup`을 사용하여 파싱하는 클래스입니다. 이 로더는 `urllib`을 사용하여 웹 페이지를 로드하고, `BeautifulSoup`을 사용하여 HTML을 파싱합니다. 사용자는 웹 페이지의 경로, 헤더 템플릿, SSL 검증 여부, 프록시 설정, 실패 시 계속 진행 여부, 인코딩 자동 설정, 인코딩 지정, 요청당 초당 요청 수, 기본 파서, 요청에 대한 추가 인자, 상태 코드에 대한 예외 발생 여부, `BeautifulSoup`의 `get_text` 메소드와 관련된 인자, 그리고 `BeautifulSoup` 생성자에 전달할 추가 인자를 설정할 수 있습니다.

다음은 `WebBaseLoader`의 기본 사용 예시입니다:

```python
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()
```

이 예시에서는 `bs4.SoupStrainer`를 사용하여 특정 HTML 클래스를 가진 요소만 파싱하도록 설정하고 있습니다. 이를 통해 전체 HTML 문서 대신 원하는 부분만 추출할 수 있습니다. `WebBaseLoader`의 인스턴스를 생성할 때 `web_paths` 매개변수에 로드하고자 하는 웹 페이지의 URL을 지정합니다. `bs_kwargs` 매개변수에는

In [ ]:
answer = rag_chain.invoke(
    "retriever 의 get_relevant_documents 기능은 어떻게 사용하나요?"
)

In [ ]:
answer = rag_chain.invoke("Agent 에서 TavilySearch 도구를 사용하는 예시를 보여주세요.")